In [2]:
# import all required libraries
import sys, os
import numpy as np
import pandas as pd
import random
from random import shuffle, choice
import time
import os
import glob
#import keras
import tensorflow as tf
#from tensorflow.keras.utils import np_utils
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.models import load_model
from tensorflow.keras import regularizers
from random import shuffle, choice
from sklearn.preprocessing import MinMaxScaler
import sklearn.metrics as metrics
from sklearn.metrics import log_loss
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import Model
from sklearn.preprocessing import MinMaxScaler,StandardScaler

# define a function to build a CNN for the SNP data.
def create_cnn(xtest, regularizer=None):
  # obtain the input dimensions.
  inputShape = (xtest.shape[1], xtest.shape[2])
  inputs = Input(shape=inputShape)
  x = inputs
  # first convolutional layer, remember to remove bias if you are intercalating with batch normalization.
  x = Conv1D(256, kernel_size=3, activation='relu', use_bias=False)(x)
  # batch normalization.
  x = BatchNormalization()(x)
  # second layer.
  x = Conv1D(256, kernel_size=3, use_bias=False, activation='relu')(x)
  x = BatchNormalization()(x)
  # third layer.
  x = Conv1D(256, kernel_size=3, use_bias=False, activation='relu')(x)
  x = BatchNormalization()(x)
  # pool the CNN outputs.
  x = GlobalMaxPooling1D()(x)
  # this part is similar to the MLP, a fully connected neural network. We intercalated with dropout to reduce overfitting.
  x = Dense(128, activation='relu')(x)
  # dropout.
  x = Dropout(0.5)(x)
  # second layer of the fully connected neural network.
  x = Dense(128, activation='relu')(x)
  x = Dropout(0.5)(x)
  # third layer of the fully connected neural network. This one matches the number of nodes coming out of the MLP.
  x = Dense(64, activation='relu')(x)
  # Construct the CNN
  #x = BatchNormalization()(x)#Not working very well
  #x = LayerNormalization()(x)#Better?
  model = Model(inputs, x)
  # Return the CNN
  return model

# define a function to combine the outputs of the traits and the SNPS.
# this was modified from: https://towardsdatascience.com/neural-networks-ensemble-33f33bea7df3
class LinearW(Layer):
    def __init__(self, init_traits_contribution=None, **kwargs):
        """
        Initializes the LinearW layer.

        Args:
            initial_traits_contribution (float, optional): If provided, sets the initial
                contribution for the first input (assumed to be MLP).
                Must be between 0 and 1. The remaining contribution is for the second input.
                If None, weights are initialized uniformly and are trainable by default.
            **kwargs: Standard Keras Layer arguments.
        """
        super(LinearW, self).__init__(**kwargs)
        self.init_traits_contribution = init_traits_contribution

        if self.init_traits_contribution is not None:
            if not (0 <= self.init_traits_contribution <= 1):
                raise ValueError("initial_mlp_contribution must be between 0 and 1.")
            # If initial_mlp_contribution is given, automatically set trainable_contributions to False
            self._trainable_contributions = False
            if self.init_traits_contribution == 0 or self.init_traits_contribution == 1:
                print("Warning: Setting initial_mlp_contribution to 0 or 1 might lead to issues with log if not handled carefully for other contributions.")
        else:
            # If no initial_mlp_contribution is given, weights are trainable by default
            self._trainable_contributions = True

    def build(self, input_shape):
        num_inputs = len(input_shape)
        if num_inputs != 2:
            raise ValueError("LinearW modified for differential trait contribution expects exactly 2 inputs (MLP, CNN).")

        if self.init_traits_contribution is not None:
            desired_contributions = np.array([self.init_traits_contribution, 1.0 - self.init_traits_contribution])
            epsilon = 1e-7 # Small epsilon for numerical stability with log
            pre_softmax_weights = np.log(desired_contributions + epsilon)
            initializer_value = pre_softmax_weights.reshape(1, 1, num_inputs)
            initializer = tf.constant_initializer(initializer_value)
        else:
            initializer = 'uniform'

        self.W = self.add_weight(
            name='linear_weights',
            shape=(1, 1, num_inputs),
            initializer=initializer,
            dtype=tf.float32,
            trainable=self._trainable_contributions # Use the internal flag
        )
        super(LinearW, self).build(input_shape)

    def call(self, inputs):
        if not isinstance(inputs, list) or len(inputs) != 2:
            raise ValueError("LinearW expects a list of exactly two input tensors (MLP_output, CNN_output).")

        inputs = [tf.expand_dims(i, -1) for i in inputs]
        inputs = Concatenate(axis=-1)(inputs)
        weights = tf.nn.softmax(self.W, axis=-1)
        return tf.reduce_sum(weights * inputs, axis=-1)
    
def linearw_contributions(model, layer_name=None, labels=("trats", "SNPs")):
    # 1) find the layer
    layer = model.get_layer(layer_name) if layer_name else next(
        (L for L in model.layers if L.__class__.__name__ == "LinearW"), None
    )
    if layer is None:
        raise ValueError("LinearW layer not found (pass layer_name= if there are several).")

    # 2) softmax over the first weight array
    weights = layer.get_weights()
    if not weights:
        raise ValueError(f"Layer '{layer.name}' has no weights.")
    logits = np.ravel(weights[0])  # e.g., shape (2,)
    contrib = tf.nn.softmax(logits).numpy()

    # 3) print nicely (if labels match length)
    if labels and len(labels) == contrib.size:
        print(", ".join(f"{lab}: {v:.4f}" for lab, v in zip(labels, contrib)))
    else:
        print("Contributions:", contrib)

    return contrib  # 1D np.ndarray

class GatedConcatenate(Layer):
    """
    Applies a trainable or fixed gate (weight) to each input branch
    before concatenating them.
    
    Args:
        initial_traits_weight (float): The starting weight for the first input (traits).
                                     Must be between 0 and 1. The weight for the
                                     second input (SNPs) will be (1 - this value).
        trainable_gates (bool): If True, the model can learn to adjust these
                                weights. If False, the weights are fixed.
    """
    def __init__(self, initial_traits_weight, trainable_gates=True, **kwargs):
        super(GatedConcatenate, self).__init__(**kwargs)
        if not (0 <= initial_traits_weight <= 1):
            raise ValueError("initial_traits_weight must be between 0 and 1.")
            
        self.initial_weights = [initial_traits_weight, 1.0 - initial_traits_weight]
        self.trainable_gates = trainable_gates

    def build(self, input_shape):
        # Create the gate variables. They are shaped for broadcasting across the features.
        self.gates = self.add_weight(
            name='gates',
            shape=(1, len(input_shape)), # Shape will be (1, 2)
            initializer=tf.constant_initializer(self.initial_weights),
            trainable=self.trainable_gates,
            dtype=tf.float32
        )
        super(GatedConcatenate, self).build(input_shape)

    def call(self, inputs):
        if not isinstance(inputs, list) or len(inputs) != 2:
            raise ValueError("GatedConcatenate expects a list of exactly two input tensors.")
        
        # Apply the gates (weights) to each branch using element-wise multiplication
        gated_traits = inputs[0] * self.gates[0, 0]
        gated_snps = inputs[1] * self.gates[0, 1]
        
        # Concatenate the scaled branches
        return Concatenate()([gated_traits, gated_snps])

    def get_config(self):
        # Needed for saving/loading the model
        config = super().get_config()
        config.update({
            'initial_traits_weight': self.initial_weights[0],
            'trainable_gates': self.trainable_gates,
        })
        return config
    
def gated_contributions(model, layer_name=None, labels=("trats", "SNPs")):
    # 1) find the layer
    gated_layer = model.get_layer('gated_concatenate')
    weights = gated_layer.get_weights()[0][0]
    rel_weight = np.sum(np.abs(weights))
    print(f"Final learned weights: Traits={weights[0]/rel_weight:.4f}, SNPs={weights[1]/rel_weight:.4f}")

In [3]:
## define variables that will be used to train all networks.
# size of the minibatches containing simulations are passed through the network in each epoch.
batch_size = 256
# number of training iterations (epochs) for the SNP only and the combined networks.
epochs = 100
# number of scenarios being classified.
num_classes = 3

In [7]:
# load the traits simulated under the BM model for the 3 scenarios. 
traits_BM = []
traits_BM = np.loadtxt("./traits/traits_BM.txt").reshape(30000,-1,100)
# transform into a NumPy array. 
traits_BM = np.array(traits_BM)

# standard scale the continuous (BM) traits
scalers_BM = {}
for i in range(traits_BM.shape[2]):
    scalers_BM[i] = StandardScaler(copy=False)
    traits_BM[:, :, i] = scalers_BM[i].fit_transform(traits_BM[:, :, i]) 

# transform into a NumPy array. 
Traits_BM = np.array(traits_BM)

# load the SNPs simulated for the 3 scenarios. 
u1 = np.load("./trainingSims/Model_1sp.npz",mmap_mode='r')
u2 = np.load("./trainingSims/Model_2sp.npz",mmap_mode='r')
u3 = np.load("./trainingSims/Model_3sp.npz",mmap_mode='r')

# combine the loaded SNPs in a single NumPy array.
u=np.concatenate((u1['Model_1sp'],u2['Model_2sp'],u3['Model_3sp']),axis=0)

# transform SNP major alleles in -1 and minor in 1.
for arr,array in enumerate(u):
    for idx,row in enumerate(array):
        if np.count_nonzero(row) > len(row)/2:
            u[arr][idx][u[arr][idx] == 1] = -1
            u[arr][idx][u[arr][idx] == 0] = 1
        else:
            u[arr][idx][u[arr][idx] == 0] = -1
            
# create a label vector in the same order as the simulations.
y=[0 for i in range(len(u1['Model_1sp']))]
y.extend([1 for i in range(len(u2['Model_2sp']))])
y.extend([2 for i in range(len(u3['Model_3sp']))])
y = np.array(y)

# make sure labels, SNP and traits matrices all have the same length.
print (len(y), len(u), len(traits_BM))

30000 30000 30000


In [4]:
################################################################################################################################################
# We will start with traits simulated under the BM model.
################################################################################################################################################

# Since we will run the analysis on several subsets, define a function for training on each data subsets (Combined datasets, SNP only and BM traits only).

# function to train on the combined datasets
def combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # reshape the traits matrices to input them into the MLP
    #traits_BM_train=traits_BM_train.reshape((traits_BM_train.shape[0], (traits_BM_train.shape[1]*traits_BM_train.shape[2])))
    #traits_BM_test=traits_BM_test.reshape((traits_BM_test.shape[0], (traits_BM_test.shape[1]*traits_BM_test.shape[2])))
    traits_BM_train=np.swapaxes(traits_BM_train, 1, 2)
    traits_BM_test=np.swapaxes(traits_BM_test, 1, 2)
    #xtrain=xtrain.reshape(*xtrain.shape[:-1], xtrain.shape[-1]//2, 2).mean(axis=-1)
    #xtest=xtest.reshape(*xtest.shape[:-1], xtest.shape[-1]//2, 2).mean(axis=-1)
    # Create the two CNNs and the combined models
    traits = create_cnn(traits_BM_train)
    snps = create_cnn(xtrain)

    #combinedInput = LinearW()([traits.output, snps.output])
    # Use the gated concatenation layer. Start with an 50/50 split but let the model learn.
    combinedInput = GatedConcatenate(
        initial_traits_weight=0.5, 
        trainable_gates=True,
        name="gated_concatenate"
    )([traits.output, snps.output])

    # The final fully-connected layer head will have two dense layers (one relu and one softmax)
    x = Dense(64, activation="relu")(combinedInput)
    #x = Dropout(0.5)(x)
    x = Dense(num_classes, activation="softmax")(x)

    # The final model accepts numerical data on the MLP input and images on the CNN input, outputting a single value
    model = Model(inputs=[traits.input, snps.input], outputs=x)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())
    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit([traits_BM_train, xtrain], ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=([traits_BM_test, xtest], ytest),callbacks=[earlyStopping])

    # Find the layer in the trained model
    #linearw_contributions(model)
    gated_contributions(model)
    
    return model

# function to train on the SNP only datasets
def SNP_subset(ytrain, ytest, xtrain, xtest):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]

    # Create the the CNN 
    snps = create_cnn(xtest)
    
    #Create the last layer for the SNP network
    #x = Dropout(0.5)(snps.output)
    xSNP = Dense(num_classes, activation="softmax")(snps.output)
    model = Model(inputs=snps.input, outputs=xSNP)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit(xtrain, ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(xtest, ytest),callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')
    
    return model

# function to train on the BM trait only datasets
def BM_subset(ytrain, ytest, traits_BM_train, traits_BM_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # reshape the traits matrices to input them into the MLP
    #traits_BM_train=traits_BM_train.reshape((traits_BM_train.shape[0], (traits_BM_train.shape[1]*traits_BM_train.shape[2])))
    #traits_BM_test=traits_BM_test.reshape((traits_BM_test.shape[0], (traits_BM_test.shape[1]*traits_BM_test.shape[2])))
    traits_BM_train=np.swapaxes(traits_BM_train, 1, 2)
    traits_BM_test=np.swapaxes(traits_BM_test, 1, 2)
    trait = create_cnn(traits_BM_train)
    #Create the last layer for the traits network

    #x = Dropout(0.5)(trait.output)
    xTRAIT = Dense(num_classes, activation="softmax")(trait.output)
    model = Model(inputs=trait.input, outputs=xTRAIT)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)
    # fit the model and record running times
    start = time.time()
    model.fit(traits_BM_train, ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(traits_BM_test, ytest),callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')

    return model

In [5]:
################################################################################################################################################
#20% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(X.shape[1]*X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
    indices_2d = np.random.choice(X.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(X.shape[2], size=missD, replace=True)
    X[i, indices_2d, indices_3d] = 0

In [6]:
################################################################################################################################################
#100 BM, 1,000 SNPs
################################################################################################################################################
# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test  = train_test_split(y,X,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_SNPs_20_Trained_Comb_Model_100BM_50SNPs.mod')

Model: "model_2"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d (Conv1D)                 (None, 98, 256)      23040       input_1[0][0]                    
__________________________________________________________________________________________________
conv1d_3 (Conv1D)               (None, 998, 256)     46080       input_2[0][0]                    
____________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 250ms/step - loss: 0.0184 - accuracy: 0.9950 - val_loss: 0.0111 - val_accuracy: 0.9985
Epoch 15/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0163 - accuracy: 0.9954 - val_loss: 0.0104 - val_accuracy: 0.9987
Epoch 16/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0137 - accuracy: 0.9959 - val_loss: 0.0097 - val_accuracy: 0.9988
Epoch 17/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0125 - accuracy: 0.9967 - val_loss: 0.0094 - val_accuracy: 0.9989
Epoch 18/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0137 - accuracy: 0.9963 - val_loss: 0.0096 - val_accuracy: 0.9989
Epoch 19/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0112 - accuracy: 0.9967 - val_loss: 0.0093 - val_accuracy: 0.9987
Epoch 20/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0090 - accuracy: 0.9977 - val_loss: 0.0088 - val_ac

In [7]:
################################################################################################################################################
#1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest  = train_test_split(y,X,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = SNP_subset(ytrain, ytest, xtrain, xtest)

# save the model
model.save(filepath='./Trained_Models/MD_SNPs_20_Trained_CNN_Model_50SNPs.mod')

Model: "model_4"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_3 (InputLayer)         [(None, 1000, 60)]        0         
_________________________________________________________________
conv1d_6 (Conv1D)            (None, 998, 256)          46080     
_________________________________________________________________
batch_normalization_6 (Batch (None, 998, 256)          1024      
_________________________________________________________________
conv1d_7 (Conv1D)            (None, 996, 256)          196608    
_________________________________________________________________
batch_normalization_7 (Batch (None, 996, 256)          1024      
_________________________________________________________________
conv1d_8 (Conv1D)            (None, 994, 256)          196608    
_________________________________________________________________
batch_normalization_8 (Batch (None, 994, 256)          1024

88/88 [==============================] - 20s 228ms/step - loss: 0.0085 - accuracy: 0.9983 - val_loss: 1.7451e-04 - val_accuracy: 1.0000
Epoch 100/100
88/88 [==============================] - 20s 228ms/step - loss: 0.0084 - accuracy: 0.9977 - val_loss: 1.5656e-04 - val_accuracy: 1.0000
Time: 2007.160490512848
INFO:tensorflow:Assets written to: ./Trained_Models/MD_SNPs_20_Trained_CNN_Model_50SNPs.mod/assets


In [8]:
################################################################################################################################################
# 40% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(X.shape[1]*X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
    indices_2d = np.random.choice(X.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(X.shape[2], size=missD, replace=True)
    X[i, indices_2d, indices_3d] = 0

In [9]:
################################################################################################################################################
#100 BM, 1,000 SNPs
################################################################################################################################################
# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test  = train_test_split(y,X,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_SNPs_40_Trained_Comb_Model_100BM_50SNPs.mod')

Model: "model_7"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_4 (InputLayer)            [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_5 (InputLayer)            [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_9 (Conv1D)               (None, 98, 256)      23040       input_4[0][0]                    
__________________________________________________________________________________________________
conv1d_12 (Conv1D)              (None, 998, 256)     46080       input_5[0][0]                    
____________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0090 - accuracy: 0.9975 - val_loss: 0.0017 - val_accuracy: 0.9996
Epoch 15/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0086 - accuracy: 0.9980 - val_loss: 0.0014 - val_accuracy: 0.9996
Epoch 16/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0081 - accuracy: 0.9975 - val_loss: 0.0013 - val_accuracy: 0.9996
Epoch 17/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0077 - accuracy: 0.9979 - val_loss: 0.0014 - val_accuracy: 0.9996
Epoch 18/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0056 - accuracy: 0.9984 - val_loss: 0.0013 - val_accuracy: 0.9997
Epoch 19/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0050 - accuracy: 0.9988 - val_loss: 8.9096e-04 - val_accuracy: 0.9997
Epoch 20/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0053 - accuracy: 0.9986 - val_loss: 9.2088e-04 

In [10]:
################################################################################################################################################
#1,000 SNPs
################################################################################################################################################
# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest = train_test_split(y,X,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = SNP_subset(ytrain, ytest, xtrain, xtest)

# save the model
model.save(filepath='./Trained_Models/MD_SNPs_40_Trained_CNN_Model_50SNPs.mod')

Model: "model_9"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_6 (InputLayer)         [(None, 1000, 60)]        0         
_________________________________________________________________
conv1d_15 (Conv1D)           (None, 998, 256)          46080     
_________________________________________________________________
batch_normalization_15 (Batc (None, 998, 256)          1024      
_________________________________________________________________
conv1d_16 (Conv1D)           (None, 996, 256)          196608    
_________________________________________________________________
batch_normalization_16 (Batc (None, 996, 256)          1024      
_________________________________________________________________
conv1d_17 (Conv1D)           (None, 994, 256)          196608    
_________________________________________________________________
batch_normalization_17 (Batc (None, 994, 256)          1024

88/88 [==============================] - 20s 228ms/step - loss: 0.0125 - accuracy: 0.9969 - val_loss: 3.1478e-04 - val_accuracy: 1.0000
Epoch 98/100
88/88 [==============================] - 20s 228ms/step - loss: 0.0123 - accuracy: 0.9970 - val_loss: 3.2545e-04 - val_accuracy: 1.0000
Epoch 99/100
88/88 [==============================] - 20s 227ms/step - loss: 0.0131 - accuracy: 0.9968 - val_loss: 3.0962e-04 - val_accuracy: 1.0000
Epoch 100/100
88/88 [==============================] - 20s 226ms/step - loss: 0.0112 - accuracy: 0.9972 - val_loss: 2.9453e-04 - val_accuracy: 1.0000
Time: 2003.9142880439758
INFO:tensorflow:Assets written to: ./Trained_Models/MD_SNPs_40_Trained_CNN_Model_50SNPs.mod/assets


In [11]:
################################################################################################################################################
# 60% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(X.shape[1]*X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
    indices_2d = np.random.choice(X.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(X.shape[2], size=missD, replace=True)
    X[i, indices_2d, indices_3d] = 0

In [12]:
################################################################################################################################################
#100 BM, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test  = train_test_split(y,X,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_SNPs_60_Trained_Comb_Model_100BM_50SNPs.mod')

Model: "model_12"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_7 (InputLayer)            [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_8 (InputLayer)            [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_18 (Conv1D)              (None, 98, 256)      23040       input_7[0][0]                    
__________________________________________________________________________________________________
conv1d_21 (Conv1D)              (None, 998, 256)     46080       input_8[0][0]                    
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0223 - accuracy: 0.9938 - val_loss: 0.0091 - val_accuracy: 0.9977
Epoch 15/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0186 - accuracy: 0.9948 - val_loss: 0.0086 - val_accuracy: 0.9980
Epoch 16/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0161 - accuracy: 0.9956 - val_loss: 0.0088 - val_accuracy: 0.9981
Epoch 17/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0141 - accuracy: 0.9967 - val_loss: 0.0076 - val_accuracy: 0.9979
Epoch 18/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0124 - accuracy: 0.9970 - val_loss: 0.0075 - val_accuracy: 0.9980
Epoch 19/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0108 - accuracy: 0.9972 - val_loss: 0.0076 - val_accuracy: 0.9981
Epoch 20/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0097 - accuracy: 0.9975 - val_loss: 0.0067 - val_ac

In [13]:
################################################################################################################################################
#1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest = train_test_split(y,X,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = SNP_subset(ytrain, ytest, xtrain, xtest)

# save the model
model.save(filepath='./Trained_Models/MD_SNPs_60_Trained_CNN_Model_50SNPs.mod')

Model: "model_14"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_9 (InputLayer)         [(None, 1000, 60)]        0         
_________________________________________________________________
conv1d_24 (Conv1D)           (None, 998, 256)          46080     
_________________________________________________________________
batch_normalization_24 (Batc (None, 998, 256)          1024      
_________________________________________________________________
conv1d_25 (Conv1D)           (None, 996, 256)          196608    
_________________________________________________________________
batch_normalization_25 (Batc (None, 996, 256)          1024      
_________________________________________________________________
conv1d_26 (Conv1D)           (None, 994, 256)          196608    
_________________________________________________________________
batch_normalization_26 (Batc (None, 994, 256)          102

88/88 [==============================] - 20s 227ms/step - loss: 0.0123 - accuracy: 0.9966 - val_loss: 0.0016 - val_accuracy: 0.9996
Epoch 99/100
88/88 [==============================] - 20s 228ms/step - loss: 0.0111 - accuracy: 0.9973 - val_loss: 0.0016 - val_accuracy: 0.9996
Epoch 100/100
88/88 [==============================] - 20s 227ms/step - loss: 0.0145 - accuracy: 0.9960 - val_loss: 0.0015 - val_accuracy: 0.9997
Time: 2006.4744839668274
INFO:tensorflow:Assets written to: ./Trained_Models/MD_SNPs_60_Trained_CNN_Model_50SNPs.mod/assets


In [14]:
################################################################################################################################################
#20% Missing Data BM
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(traits_BM.shape[1]*traits_BM.shape[2]*(missD_perc/100))
for i in range(traits_BM.shape[0]):
    indices_2d = np.random.choice(traits_BM.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_BM.shape[2], size=missD, replace=True)
    traits_BM[i, indices_2d, indices_3d] = 0

In [15]:
################################################################################################################################################
#100 BM, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test  = train_test_split(y,X,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_BM_20_Trained_Comb_Model_100BM_50SNPs.mod')

Model: "model_17"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_10 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_11 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_27 (Conv1D)              (None, 98, 256)      23040       input_10[0][0]                   
__________________________________________________________________________________________________
conv1d_30 (Conv1D)              (None, 998, 256)     46080       input_11[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0060 - accuracy: 0.9982 - val_loss: 4.6080e-04 - val_accuracy: 0.9999
Epoch 15/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0047 - accuracy: 0.9990 - val_loss: 3.8500e-04 - val_accuracy: 0.9999
Epoch 16/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0041 - accuracy: 0.9992 - val_loss: 4.0799e-04 - val_accuracy: 0.9999
Epoch 17/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0049 - accuracy: 0.9984 - val_loss: 3.8363e-04 - val_accuracy: 0.9999
Epoch 18/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0039 - accuracy: 0.9992 - val_loss: 4.1385e-04 - val_accuracy: 0.9999
Epoch 19/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0036 - accuracy: 0.9992 - val_loss: 2.9863e-04 - val_accuracy: 0.9999
Epoch 20/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0038 - accuracy: 0.9990 - v

In [16]:
################################################################################################################################################
#100 BM
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_BM_train, traits_BM_test  = train_test_split(y,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = BM_subset(ytrain, ytest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_BM_20_Trained_Traits_Model_100BM.mod')

Model: "model_19"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_12 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_33 (Conv1D)           (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_33 (Batc (None, 98, 256)           1024      
_________________________________________________________________
conv1d_34 (Conv1D)           (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_34 (Batc (None, 96, 256)           1024      
_________________________________________________________________
conv1d_35 (Conv1D)           (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_35 (Batc (None, 94, 256)           102

Epoch 43/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0036 - accuracy: 0.9992 - val_loss: 0.0156 - val_accuracy: 0.9969
Epoch 44/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0044 - accuracy: 0.9986 - val_loss: 0.0157 - val_accuracy: 0.9971
Epoch 45/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0034 - accuracy: 0.9992 - val_loss: 0.0160 - val_accuracy: 0.9969
Epoch 46/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0033 - accuracy: 0.9992 - val_loss: 0.0155 - val_accuracy: 0.9969
Epoch 47/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0034 - accuracy: 0.9992 - val_loss: 0.0153 - val_accuracy: 0.9968
Epoch 48/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0027 - accuracy: 0.9996 - val_loss: 0.0168 - val_accuracy: 0.9969
Epoch 49/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0031 - accuracy: 0.9992 - val_loss: 0.0166 - val_accuracy: 0.9969

In [17]:
################################################################################################################################################
#40% Missing Data BM
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(traits_BM.shape[1]*traits_BM.shape[2]*(missD_perc/100))
for i in range(traits_BM.shape[0]):
    indices_2d = np.random.choice(traits_BM.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_BM.shape[2], size=missD, replace=True)
    traits_BM[i, indices_2d, indices_3d] = 0

In [18]:
################################################################################################################################################
#100 BM, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test  = train_test_split(y,X,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_BM_40_Trained_Comb_Model_100BM_50SNPs.mod')

Model: "model_22"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_13 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_14 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_36 (Conv1D)              (None, 98, 256)      23040       input_13[0][0]                   
__________________________________________________________________________________________________
conv1d_39 (Conv1D)              (None, 998, 256)     46080       input_14[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0071 - accuracy: 0.9980 - val_loss: 8.8016e-05 - val_accuracy: 1.0000
Epoch 15/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0071 - accuracy: 0.9980 - val_loss: 8.1054e-05 - val_accuracy: 1.0000
Epoch 16/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0058 - accuracy: 0.9985 - val_loss: 7.9886e-05 - val_accuracy: 1.0000
Epoch 17/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0066 - accuracy: 0.9984 - val_loss: 9.2144e-05 - val_accuracy: 1.0000
Epoch 18/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0063 - accuracy: 0.9979 - val_loss: 6.3230e-05 - val_accuracy: 1.0000
Epoch 19/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0046 - accuracy: 0.9988 - val_loss: 4.9142e-05 - val_accuracy: 1.0000
Epoch 20/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0043 - accuracy: 0.9989 - v

In [19]:
################################################################################################################################################
#100 BM
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_BM_train, traits_BM_test  = train_test_split(y,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = BM_subset(ytrain, ytest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_BM_40_Trained_Traits_Model_100BM.mod')

Model: "model_24"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_15 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_42 (Conv1D)           (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_42 (Batc (None, 98, 256)           1024      
_________________________________________________________________
conv1d_43 (Conv1D)           (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_43 (Batc (None, 96, 256)           1024      
_________________________________________________________________
conv1d_44 (Conv1D)           (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_44 (Batc (None, 94, 256)           102

Epoch 43/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0043 - accuracy: 0.9991 - val_loss: 0.0264 - val_accuracy: 0.9944
Epoch 44/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0039 - accuracy: 0.9992 - val_loss: 0.0283 - val_accuracy: 0.9947
Epoch 45/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0042 - accuracy: 0.9986 - val_loss: 0.0261 - val_accuracy: 0.9947
Epoch 46/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0047 - accuracy: 0.9989 - val_loss: 0.0255 - val_accuracy: 0.9951
Epoch 47/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0037 - accuracy: 0.9993 - val_loss: 0.0264 - val_accuracy: 0.9949
Epoch 48/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0039 - accuracy: 0.9992 - val_loss: 0.0262 - val_accuracy: 0.9947
Epoch 49/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0042 - accuracy: 0.9992 - val_loss: 0.0267 - val_accuracy: 0.9947

In [20]:
################################################################################################################################################
# 60% Missing Data BM
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(traits_BM.shape[1]*traits_BM.shape[2]*(missD_perc/100))
for i in range(traits_BM.shape[0]):
    indices_2d = np.random.choice(traits_BM.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_BM.shape[2], size=missD, replace=True)
    traits_BM[i, indices_2d, indices_3d] = 0

In [21]:
################################################################################################################################################
#100 BM, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test  = train_test_split(y,X,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_BM_60_Trained_Comb_Model_100BM_50SNPs.mod')

Model: "model_27"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_16 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_17 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_45 (Conv1D)              (None, 98, 256)      23040       input_16[0][0]                   
__________________________________________________________________________________________________
conv1d_48 (Conv1D)              (None, 998, 256)     46080       input_17[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0069 - accuracy: 0.9982 - val_loss: 0.0013 - val_accuracy: 0.9995
Epoch 15/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0052 - accuracy: 0.9984 - val_loss: 0.0013 - val_accuracy: 0.9995
Epoch 16/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0046 - accuracy: 0.9989 - val_loss: 9.4843e-04 - val_accuracy: 0.9995
Epoch 17/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0046 - accuracy: 0.9988 - val_loss: 0.0010 - val_accuracy: 0.9995
Epoch 18/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0047 - accuracy: 0.9988 - val_loss: 6.3743e-04 - val_accuracy: 0.9996
Epoch 19/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0038 - accuracy: 0.9991 - val_loss: 0.0010 - val_accuracy: 0.9995
Epoch 20/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0035 - accuracy: 0.9994 - val_loss: 0.0013 

Epoch 69/100
88/88 [==============================] - 22s 248ms/step - loss: 8.7949e-04 - accuracy: 0.9998 - val_loss: 5.1617e-04 - val_accuracy: 0.9997
Epoch 70/100
88/88 [==============================] - 22s 248ms/step - loss: 7.9899e-04 - accuracy: 0.9998 - val_loss: 4.0176e-04 - val_accuracy: 0.9997
Epoch 71/100
88/88 [==============================] - 22s 248ms/step - loss: 8.6150e-04 - accuracy: 0.9998 - val_loss: 4.3321e-04 - val_accuracy: 0.9997
Epoch 72/100
88/88 [==============================] - 22s 248ms/step - loss: 6.6187e-04 - accuracy: 0.9998 - val_loss: 2.7864e-04 - val_accuracy: 0.9997
Epoch 73/100
88/88 [==============================] - 22s 248ms/step - loss: 8.4450e-04 - accuracy: 0.9997 - val_loss: 2.4925e-04 - val_accuracy: 0.9999
Epoch 74/100
88/88 [==============================] - 22s 248ms/step - loss: 6.7757e-04 - accuracy: 0.9999 - val_loss: 2.5070e-04 - val_accuracy: 0.9999
Epoch 75/100
88/88 [==============================] - 22s 248ms/step - loss: 4.708

In [22]:
################################################################################################################################################
#100 BM
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_BM_train, traits_BM_test  = train_test_split(y,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = BM_subset(ytrain, ytest, traits_BM_train, traits_BM_test)

# save the model
model.save(filepath='./Trained_Models/MD_BM_60_Trained_Traits_Model_100BM.mod')

Model: "model_29"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_18 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_51 (Conv1D)           (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_51 (Batc (None, 98, 256)           1024      
_________________________________________________________________
conv1d_52 (Conv1D)           (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_52 (Batc (None, 96, 256)           1024      
_________________________________________________________________
conv1d_53 (Conv1D)           (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_53 (Batc (None, 94, 256)           102

Epoch 43/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0066 - accuracy: 0.9986 - val_loss: 0.0341 - val_accuracy: 0.9911
Epoch 44/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0056 - accuracy: 0.9987 - val_loss: 0.0356 - val_accuracy: 0.9911
Epoch 45/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0058 - accuracy: 0.9984 - val_loss: 0.0352 - val_accuracy: 0.9909
Epoch 46/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0054 - accuracy: 0.9987 - val_loss: 0.0352 - val_accuracy: 0.9916
Epoch 47/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0063 - accuracy: 0.9984 - val_loss: 0.0354 - val_accuracy: 0.9913
Epoch 48/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0051 - accuracy: 0.9986 - val_loss: 0.0359 - val_accuracy: 0.9907
Epoch 49/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0044 - accuracy: 0.9992 - val_loss: 0.0360 - val_accuracy: 0.9915

In [23]:
################################################################################################################################################
# Now repeat with traits simulated under the OU model.
################################################################################################################################################

# Since we will run the analysis on several subsets, define a function for training on each data subsets (Combined datasets, SNP only and OU traits only).

# function to train on the combined datasets
def combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # reshape the traits matrices to input them into the MLP
    #traits_BM_train=traits_BM_train.reshape((traits_BM_train.shape[0], (traits_BM_train.shape[1]*traits_BM_train.shape[2])))
    #traits_BM_test=traits_BM_test.reshape((traits_BM_test.shape[0], (traits_BM_test.shape[1]*traits_BM_test.shape[2])))
    traits_OU_train=np.swapaxes(traits_OU_train, 1, 2)
    traits_OU_test=np.swapaxes(traits_OU_test, 1, 2)
    # Create the two CNNs and the combined models
    traits = create_cnn(traits_OU_train)
    snps = create_cnn(xtrain)
    #combinedInput = LinearW()([traits.output, snps.output])
    # Use the gated concatenation layer. Start with an 50/50 split but let the model learn.
    combinedInput = GatedConcatenate(
        initial_traits_weight=0.5, 
        trainable_gates=True,
        name="gated_concatenate"
    )([traits.output, snps.output])

    # The final fully-connected layer head will have two dense layers (one relu and one softmax)
    x = Dense(64, activation="relu")(combinedInput)
    #x = Dropout(0.5)(x)
    x = Dense(num_classes, activation="softmax")(x)

    # The final model accepts numerical data on the MLP input and images on the CNN input, outputting a single value
    model = Model(inputs=[traits.input, snps.input], outputs=x)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())
    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit([traits_OU_train, xtrain], ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=([traits_OU_test, xtest], ytest),callbacks=[earlyStopping])
    
    # Find the layer in the trained model
    #linearw_contributions(model)
    gated_contributions(model)
    print (f'Time: {time.time() - start}')
    
    return model

# function to train on the OU trait only datasets
def OU_subset(ytrain, ytest, traits_OU_train, traits_OU_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # reshape the traits matrices to input them into the MLP
    #traits_BM_train=traits_BM_train.reshape((traits_BM_train.shape[0], (traits_BM_train.shape[1]*traits_BM_train.shape[2])))
    #traits_BM_test=traits_BM_test.reshape((traits_BM_test.shape[0], (traits_BM_test.shape[1]*traits_BM_test.shape[2])))
    traits_OU_train=np.swapaxes(traits_OU_train, 1, 2)
    traits_OU_test=np.swapaxes(traits_OU_test, 1, 2)
    trait = create_cnn(traits_OU_train)
    #Create the last layer for the traits network

    #x = Dropout(0.5)(trait.output)
    xTRAIT = Dense(num_classes, activation="softmax")(trait.output)
    model = Model(inputs=trait.input, outputs=xTRAIT)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)
    # fit the model and record running times
    start = time.time()
    model.fit(traits_OU_train, ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(traits_OU_test, ytest),callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')
    
    return model

In [24]:
#remove
# load the SNPs simulated for the 3 scenarios. 
u1 = np.load("./trainingSims/Model_1sp.npz",mmap_mode='r')
u2 = np.load("./trainingSims/Model_2sp.npz",mmap_mode='r')
u3 = np.load("./trainingSims/Model_3sp.npz",mmap_mode='r')

# combine the loaded SNPs in a single NumPy array.
u=np.concatenate((u1['Model_1sp'],u2['Model_2sp'],u3['Model_3sp']),axis=0)

# transform SNP major alleles in -1 and minor in 1.
for arr,array in enumerate(u):
    for idx,row in enumerate(array):
        if np.count_nonzero(row) > len(row)/2:
            u[arr][idx][u[arr][idx] == 1] = -1
            u[arr][idx][u[arr][idx] == 0] = 1
        else:
            u[arr][idx][u[arr][idx] == 0] = -1
            
# create a label vector in the same order as the simulations.
y=[0 for i in range(len(u1['Model_1sp']))]
y.extend([1 for i in range(len(u2['Model_2sp']))])
y.extend([2 for i in range(len(u3['Model_3sp']))])
y = np.array(y)

In [25]:
# load the traits simulated under the OU model for the 3 scenarios. 
traits_OU = []
traits_OU = np.loadtxt("./traits/traits_OU.txt").reshape(30000,-1,100)
# transform into a NumPy array. 
traits_OU = np.array(traits_OU)

# standard scale the continuous (OU) traits
scalers_OU = {}
for i in range(traits_OU.shape[2]):
    scalers_OU[i] = StandardScaler(copy=False)
    traits_OU[:, :, i] = scalers_OU[i].fit_transform(traits_OU[:, :, i]) 

# transform into a NumPy array. 
Traits_OU = np.array(traits_OU)

In [26]:
################################################################################################################################################
#20% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(X.shape[1]*X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
    indices_2d = np.random.choice(X.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(X.shape[2], size=missD, replace=True)
    X[i, indices_2d, indices_3d] = 0

In [27]:
################################################################################################################################################
#100 OU, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test  = train_test_split(y,X,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_SNPs_20_Trained_Comb_Model_100OU_50SNPs.mod')

Model: "model_32"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_19 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_20 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_54 (Conv1D)              (None, 98, 256)      23040       input_19[0][0]                   
__________________________________________________________________________________________________
conv1d_57 (Conv1D)              (None, 998, 256)     46080       input_20[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0068 - accuracy: 0.9983 - val_loss: 4.4945e-04 - val_accuracy: 0.9997
Epoch 15/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0056 - accuracy: 0.9987 - val_loss: 3.7594e-04 - val_accuracy: 0.9999
Epoch 16/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0052 - accuracy: 0.9988 - val_loss: 3.8071e-04 - val_accuracy: 0.9999
Epoch 17/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0054 - accuracy: 0.9988 - val_loss: 3.8484e-04 - val_accuracy: 0.9999
Epoch 18/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0043 - accuracy: 0.9991 - val_loss: 3.1647e-04 - val_accuracy: 0.9999
Epoch 19/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0042 - accuracy: 0.9991 - val_loss: 2.9089e-04 - val_accuracy: 0.9999
Epoch 20/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0045 - accuracy: 0.9985 - v

Epoch 69/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0012 - accuracy: 0.9996 - val_loss: 9.2660e-05 - val_accuracy: 1.0000
Epoch 70/100
88/88 [==============================] - 22s 248ms/step - loss: 8.9113e-04 - accuracy: 0.9998 - val_loss: 8.1065e-05 - val_accuracy: 1.0000
Epoch 71/100
88/88 [==============================] - 22s 248ms/step - loss: 7.1997e-04 - accuracy: 0.9998 - val_loss: 6.5017e-05 - val_accuracy: 1.0000
Epoch 72/100
88/88 [==============================] - 22s 248ms/step - loss: 6.7384e-04 - accuracy: 0.9999 - val_loss: 6.2799e-05 - val_accuracy: 1.0000
Final learned weights: Traits=0.4379, SNPs=0.5621
Time: 1574.1831803321838
INFO:tensorflow:Assets written to: ./Trained_Models/MD_SNPs_20_Trained_Comb_Model_100OU_50SNPs.mod/assets


In [28]:
################################################################################################################################################
#40% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(X.shape[1]*X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
    indices_2d = np.random.choice(X.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(X.shape[2], size=missD, replace=True)
    X[i, indices_2d, indices_3d] = 0

In [29]:
################################################################################################################################################
#100 OU, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test  = train_test_split(y,X,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_SNPs_40_Trained_Comb_Model_100OU_50SNPs.mod')

Model: "model_35"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_21 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_22 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_60 (Conv1D)              (None, 98, 256)      23040       input_21[0][0]                   
__________________________________________________________________________________________________
conv1d_63 (Conv1D)              (None, 998, 256)     46080       input_22[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0178 - accuracy: 0.9956 - val_loss: 0.0051 - val_accuracy: 0.9985
Epoch 15/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0161 - accuracy: 0.9954 - val_loss: 0.0048 - val_accuracy: 0.9984
Epoch 16/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0160 - accuracy: 0.9953 - val_loss: 0.0045 - val_accuracy: 0.9987
Epoch 17/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0130 - accuracy: 0.9961 - val_loss: 0.0049 - val_accuracy: 0.9985
Epoch 18/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0116 - accuracy: 0.9968 - val_loss: 0.0046 - val_accuracy: 0.9988
Epoch 19/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0101 - accuracy: 0.9972 - val_loss: 0.0047 - val_accuracy: 0.9989
Epoch 20/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0091 - accuracy: 0.9974 - val_loss: 0.0049 - val_ac

In [30]:
################################################################################################################################################
#60% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(X.shape[1]*X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
    indices_2d = np.random.choice(X.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(X.shape[2], size=missD, replace=True)
    X[i, indices_2d, indices_3d] = 0

In [31]:
################################################################################################################################################
#100 OU, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test  = train_test_split(y,X,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_SNPs_60_Trained_Comb_Model_100OU_50SNPs.mod')

Model: "model_38"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_23 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_24 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_66 (Conv1D)              (None, 98, 256)      23040       input_23[0][0]                   
__________________________________________________________________________________________________
conv1d_69 (Conv1D)              (None, 998, 256)     46080       input_24[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0101 - accuracy: 0.9976 - val_loss: 0.0021 - val_accuracy: 0.9995
Epoch 15/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0084 - accuracy: 0.9979 - val_loss: 0.0018 - val_accuracy: 0.9995
Epoch 16/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0078 - accuracy: 0.9983 - val_loss: 0.0016 - val_accuracy: 0.9995
Epoch 17/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0064 - accuracy: 0.9981 - val_loss: 0.0016 - val_accuracy: 0.9995
Epoch 18/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0049 - accuracy: 0.9988 - val_loss: 0.0015 - val_accuracy: 0.9995
Epoch 19/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0049 - accuracy: 0.9990 - val_loss: 0.0015 - val_accuracy: 0.9996
Epoch 20/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0055 - accuracy: 0.9984 - val_loss: 0.0013 - val_ac

In [32]:
################################################################################################################################################
# 20% Missing Data OU
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(traits_OU.shape[1]*traits_OU.shape[2]*(missD_perc/100))
for i in range(traits_OU.shape[0]):
    indices_2d = np.random.choice(traits_OU.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_OU.shape[2], size=missD, replace=True)
    traits_OU[i, indices_2d, indices_3d] = 0

In [33]:
################################################################################################################################################
#100 OU, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test  = train_test_split(y,X,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_OU_20_Trained_Comb_Model_100OU_50SNPs.mod')

Model: "model_41"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_25 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_26 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_72 (Conv1D)              (None, 98, 256)      23040       input_25[0][0]                   
__________________________________________________________________________________________________
conv1d_75 (Conv1D)              (None, 998, 256)     46080       input_26[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0073 - accuracy: 0.9980 - val_loss: 8.7938e-04 - val_accuracy: 0.9999
Epoch 15/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0057 - accuracy: 0.9986 - val_loss: 8.1091e-04 - val_accuracy: 0.9999
Epoch 16/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0063 - accuracy: 0.9985 - val_loss: 8.2510e-04 - val_accuracy: 0.9999
Epoch 17/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0054 - accuracy: 0.9988 - val_loss: 7.5868e-04 - val_accuracy: 0.9999
Epoch 18/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0049 - accuracy: 0.9988 - val_loss: 8.5453e-04 - val_accuracy: 0.9999
Epoch 19/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0049 - accuracy: 0.9988 - val_loss: 7.2004e-04 - val_accuracy: 0.9999
Epoch 20/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0040 - accuracy: 0.9993 - v

In [35]:
################################################################################################################################################
#100 OU
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_OU_train, traits_OU_test  = train_test_split(y,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = OU_subset(ytrain, ytest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_OU_20_Trained_Traits_Model_100OU.mod')

Model: "model_43"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_27 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_78 (Conv1D)           (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_78 (Batc (None, 98, 256)           1024      
_________________________________________________________________
conv1d_79 (Conv1D)           (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_79 (Batc (None, 96, 256)           1024      
_________________________________________________________________
conv1d_80 (Conv1D)           (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_80 (Batc (None, 94, 256)           102

Epoch 43/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0034 - accuracy: 0.9990 - val_loss: 0.0072 - val_accuracy: 0.9983
Epoch 44/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0032 - accuracy: 0.9993 - val_loss: 0.0073 - val_accuracy: 0.9983
Epoch 45/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0041 - accuracy: 0.9992 - val_loss: 0.0075 - val_accuracy: 0.9983
Epoch 46/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0034 - accuracy: 0.9993 - val_loss: 0.0074 - val_accuracy: 0.9983
Epoch 47/100
88/88 [==============================] - 2s 18ms/step - loss: 0.0036 - accuracy: 0.9990 - val_loss: 0.0076 - val_accuracy: 0.9987
Epoch 48/100
88/88 [==============================] - 2s 17ms/step - loss: 0.0036 - accuracy: 0.9993 - val_loss: 0.0072 - val_accuracy: 0.9984
Epoch 49/100
88/88 [==============================] - 1s 17ms/step - loss: 0.0030 - accuracy: 0.9994 - val_loss: 0.0072 - val_accuracy: 0.9983

In [36]:
################################################################################################################################################
# 40% Missing Data OU
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(traits_OU.shape[1]*traits_OU.shape[2]*(missD_perc/100))
for i in range(traits_OU.shape[0]):
    indices_2d = np.random.choice(traits_OU.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_OU.shape[2], size=missD, replace=True)
    traits_OU[i, indices_2d, indices_3d] = 0

In [37]:
################################################################################################################################################
#100 OU, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test  = train_test_split(y,X,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_OU_40_Trained_Comb_Model_100OU_50SNPs.mod')

Model: "model_46"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_28 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_29 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_81 (Conv1D)              (None, 98, 256)      23040       input_28[0][0]                   
__________________________________________________________________________________________________
conv1d_84 (Conv1D)              (None, 998, 256)     46080       input_29[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0069 - accuracy: 0.9979 - val_loss: 9.9721e-05 - val_accuracy: 1.0000
Epoch 15/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0073 - accuracy: 0.9983 - val_loss: 9.5324e-05 - val_accuracy: 1.0000
Epoch 16/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0058 - accuracy: 0.9985 - val_loss: 1.1208e-04 - val_accuracy: 1.0000
Epoch 17/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0053 - accuracy: 0.9987 - val_loss: 8.6400e-05 - val_accuracy: 1.0000
Epoch 18/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0042 - accuracy: 0.9990 - val_loss: 6.8410e-05 - val_accuracy: 1.0000
Epoch 19/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0040 - accuracy: 0.9989 - val_loss: 5.1332e-05 - val_accuracy: 1.0000
Epoch 20/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0039 - accuracy: 0.9988 - v

In [38]:
################################################################################################################################################
#100 OU
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_OU_train, traits_OU_test  = train_test_split(y,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = OU_subset(ytrain, ytest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_OU_40_Trained_Traits_Model_100OU.mod')

Model: "model_48"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_30 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_87 (Conv1D)           (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_87 (Batc (None, 98, 256)           1024      
_________________________________________________________________
conv1d_88 (Conv1D)           (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_88 (Batc (None, 96, 256)           1024      
_________________________________________________________________
conv1d_89 (Conv1D)           (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_89 (Batc (None, 94, 256)           102

Epoch 43/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0045 - accuracy: 0.9989 - val_loss: 0.0130 - val_accuracy: 0.9959
Epoch 44/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0046 - accuracy: 0.9989 - val_loss: 0.0134 - val_accuracy: 0.9957
Epoch 45/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0052 - accuracy: 0.9988 - val_loss: 0.0127 - val_accuracy: 0.9960
Epoch 46/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0046 - accuracy: 0.9987 - val_loss: 0.0135 - val_accuracy: 0.9956
Epoch 47/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0051 - accuracy: 0.9988 - val_loss: 0.0124 - val_accuracy: 0.9959
Epoch 48/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0041 - accuracy: 0.9992 - val_loss: 0.0142 - val_accuracy: 0.9959
Epoch 49/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0045 - accuracy: 0.9986 - val_loss: 0.0128 - val_accuracy: 0.9961

In [39]:
################################################################################################################################################
# 60% Missing Data OU
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(traits_OU.shape[1]*traits_OU.shape[2]*(missD_perc/100))
for i in range(traits_OU.shape[0]):
    indices_2d = np.random.choice(traits_OU.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_OU.shape[2], size=missD, replace=True)
    traits_OU[i, indices_2d, indices_3d] = 0

In [40]:
################################################################################################################################################
#100 OU, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test  = train_test_split(y,X,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_OU_60_Trained_Comb_Model_100OU_50SNPs.mod')

Model: "model_51"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_31 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_32 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_90 (Conv1D)              (None, 98, 256)      23040       input_31[0][0]                   
__________________________________________________________________________________________________
conv1d_93 (Conv1D)              (None, 998, 256)     46080       input_32[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0074 - accuracy: 0.9980 - val_loss: 6.3820e-04 - val_accuracy: 0.9999
Epoch 15/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0068 - accuracy: 0.9980 - val_loss: 5.4206e-04 - val_accuracy: 0.9999
Epoch 16/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0051 - accuracy: 0.9988 - val_loss: 5.3616e-04 - val_accuracy: 0.9999
Epoch 17/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0046 - accuracy: 0.9984 - val_loss: 6.4690e-04 - val_accuracy: 0.9999
Epoch 18/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0039 - accuracy: 0.9992 - val_loss: 5.2810e-04 - val_accuracy: 0.9999
Epoch 19/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0038 - accuracy: 0.9991 - val_loss: 4.6834e-04 - val_accuracy: 0.9999
Epoch 20/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0045 - accuracy: 0.9988 - v

Epoch 69/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0012 - accuracy: 0.9996 - val_loss: 7.0597e-05 - val_accuracy: 1.0000
Epoch 70/100
88/88 [==============================] - 22s 247ms/step - loss: 9.6370e-04 - accuracy: 0.9997 - val_loss: 6.2150e-05 - val_accuracy: 1.0000
Epoch 71/100
88/88 [==============================] - 22s 247ms/step - loss: 9.2225e-04 - accuracy: 0.9997 - val_loss: 5.1051e-05 - val_accuracy: 1.0000
Epoch 72/100
88/88 [==============================] - 22s 247ms/step - loss: 7.8717e-04 - accuracy: 0.9997 - val_loss: 4.5479e-05 - val_accuracy: 1.0000
Epoch 73/100
88/88 [==============================] - 22s 246ms/step - loss: 7.2381e-04 - accuracy: 0.9998 - val_loss: 3.9800e-05 - val_accuracy: 1.0000
Epoch 74/100
88/88 [==============================] - 22s 247ms/step - loss: 7.0310e-04 - accuracy: 0.9999 - val_loss: 5.8690e-05 - val_accuracy: 1.0000
Epoch 75/100
88/88 [==============================] - 22s 247ms/step - loss: 6.9177e-0

In [41]:
################################################################################################################################################
#100 OU
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_OU_train, traits_OU_test  = train_test_split(y,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = OU_subset(ytrain, ytest, traits_OU_train, traits_OU_test)

# save the model
model.save(filepath='./Trained_Models/MD_OU_60_Trained_Traits_Model_100OU.mod')

Model: "model_53"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_33 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_96 (Conv1D)           (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_96 (Batc (None, 98, 256)           1024      
_________________________________________________________________
conv1d_97 (Conv1D)           (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_97 (Batc (None, 96, 256)           1024      
_________________________________________________________________
conv1d_98 (Conv1D)           (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_98 (Batc (None, 94, 256)           102

Epoch 43/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0052 - accuracy: 0.9988 - val_loss: 0.0256 - val_accuracy: 0.9943
Epoch 44/100
88/88 [==============================] - 1s 15ms/step - loss: 0.0045 - accuracy: 0.9991 - val_loss: 0.0266 - val_accuracy: 0.9944
Epoch 45/100
88/88 [==============================] - 1s 15ms/step - loss: 0.0046 - accuracy: 0.9987 - val_loss: 0.0258 - val_accuracy: 0.9945
Epoch 46/100
88/88 [==============================] - 1s 15ms/step - loss: 0.0049 - accuracy: 0.9988 - val_loss: 0.0259 - val_accuracy: 0.9941
Epoch 47/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0051 - accuracy: 0.9984 - val_loss: 0.0262 - val_accuracy: 0.9948
Epoch 48/100
88/88 [==============================] - 1s 15ms/step - loss: 0.0046 - accuracy: 0.9988 - val_loss: 0.0266 - val_accuracy: 0.9947
Epoch 49/100
88/88 [==============================] - 1s 15ms/step - loss: 0.0046 - accuracy: 0.9989 - val_loss: 0.0264 - val_accuracy: 0.9943

In [4]:
################################################################################################################################################
# Now repeat with discrete traits.
################################################################################################################################################

# Since we will run the analysis on several subsets, define a function for training on each data subsets (Combined datasets, SNP only and discrete traits only).

# function to train on the combined datasets
def combined_disc_subset(ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # reshape the traits matrices to input them into the MLP
    #traits_BM_train=traits_BM_train.reshape((traits_BM_train.shape[0], (traits_BM_train.shape[1]*traits_BM_train.shape[2])))
    #traits_BM_test=traits_BM_test.reshape((traits_BM_test.shape[0], (traits_BM_test.shape[1]*traits_BM_test.shape[2])))
    traits_disc_train=np.swapaxes(traits_disc_train, 1, 2)
    traits_disc_test=np.swapaxes(traits_disc_test, 1, 2)
    # Create the two CNNs and the combined models
    traits = create_cnn(traits_disc_train)
    snps = create_cnn(xtrain)
    #combinedInput = LinearW()([traits.output, snps.output])
    # Use the gated concatenation layer. Start with an 50/50 split but let the model learn.
    combinedInput = GatedConcatenate(
        initial_traits_weight=0.5, 
        trainable_gates=True,
        name="gated_concatenate"
    )([traits.output, snps.output])

    # The final fully-connected layer head will have two dense layers (one relu and one softmax)
    x = Dense(64, activation="relu")(combinedInput)
    #x = Dropout(0.5)(x)
    x = Dense(num_classes, activation="softmax")(x)

    # The final model accepts numerical data on the MLP input and images on the CNN input, outputting a single value
    model = Model(inputs=[traits.input, snps.input], outputs=x)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())
    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit([traits_disc_train, xtrain], ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=([traits_disc_test, xtest], ytest),callbacks=[earlyStopping])

    # Find the layer in the trained model
    #linearw_contributions(model)
    gated_contributions(model)    
    print (f'Time: {time.time() - start}')
    
    return model

# function to train on the discrete trait only datasets
def disc_subset(ytrain, ytest, traits_disc_train, traits_disc_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # reshape the traits matrices to input them into the MLP
    #traits_BM_train=traits_BM_train.reshape((traits_BM_train.shape[0], (traits_BM_train.shape[1]*traits_BM_train.shape[2])))
    #traits_BM_test=traits_BM_test.reshape((traits_BM_test.shape[0], (traits_BM_test.shape[1]*traits_BM_test.shape[2])))
    traits_disc_train=np.swapaxes(traits_disc_train, 1, 2)
    traits_disc_test=np.swapaxes(traits_disc_test, 1, 2)
    trait = create_cnn(traits_disc_train)
    #Create the last layer for the traits network

    #x = Dropout(0.5)(trait.output)
    xTRAIT = Dense(num_classes, activation="softmax")(trait.output)
    model = Model(inputs=trait.input, outputs=xTRAIT)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)
    # fit the model and record running times
    start = time.time()
    model.fit(traits_disc_train, ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(traits_disc_test, ytest),callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')
    
    return model

In [5]:
# load the traits simulated under the OU model for the 3 scenarios. 
traits_disc = []
traits_disc = np.loadtxt("./traits/traits_disc.txt").reshape(30000,-1,100)

# transform into a NumPy array. 
traits_disc = np.array(traits_disc)
Traits_disc = np.array(traits_disc)

In [44]:
################################################################################################################################################
#20% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_disc=np.array(Traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(X.shape[1]*X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
    indices_2d = np.random.choice(X.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(X.shape[2], size=missD, replace=True)
    X[i, indices_2d, indices_3d] = 0

In [45]:
################################################################################################################################################
#100 discrete, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test  = train_test_split(y,X,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_disc_subset(ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_SNPs_20_Trained_Comb_Model_100discrete_50SNPs.mod')

Model: "model_56"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_34 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_35 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_99 (Conv1D)              (None, 98, 256)      23040       input_34[0][0]                   
__________________________________________________________________________________________________
conv1d_102 (Conv1D)             (None, 998, 256)     46080       input_35[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0070 - accuracy: 0.9981 - val_loss: 0.0020 - val_accuracy: 0.9997
Epoch 15/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0066 - accuracy: 0.9982 - val_loss: 0.0020 - val_accuracy: 0.9997
Epoch 16/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0056 - accuracy: 0.9985 - val_loss: 0.0018 - val_accuracy: 0.9997
Epoch 17/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0052 - accuracy: 0.9987 - val_loss: 0.0018 - val_accuracy: 0.9997
Epoch 18/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0056 - accuracy: 0.9987 - val_loss: 0.0017 - val_accuracy: 0.9997
Epoch 19/100
88/88 [==============================] - 22s 246ms/step - loss: 0.0042 - accuracy: 0.9989 - val_loss: 0.0017 - val_accuracy: 0.9997
Epoch 20/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0043 - accuracy: 0.9991 - val_loss: 0.0017 - val_ac

In [46]:
################################################################################################################################################
#40% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_disc=np.array(Traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(X.shape[1]*X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
    indices_2d = np.random.choice(X.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(X.shape[2], size=missD, replace=True)
    X[i, indices_2d, indices_3d] = 0

In [47]:
################################################################################################################################################
#100 discrete, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test  = train_test_split(y,X,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_disc_subset(ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_SNPs_40_Trained_Comb_Model_100discrete_50SNPs.mod')

Model: "model_59"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_36 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_37 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_105 (Conv1D)             (None, 98, 256)      23040       input_36[0][0]                   
__________________________________________________________________________________________________
conv1d_108 (Conv1D)             (None, 998, 256)     46080       input_37[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0124 - accuracy: 0.9963 - val_loss: 4.3310e-04 - val_accuracy: 0.9999
Epoch 15/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0091 - accuracy: 0.9977 - val_loss: 3.3535e-04 - val_accuracy: 0.9999
Epoch 16/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0097 - accuracy: 0.9976 - val_loss: 2.4196e-04 - val_accuracy: 1.0000
Epoch 17/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0063 - accuracy: 0.9986 - val_loss: 1.9698e-04 - val_accuracy: 1.0000
Epoch 18/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0066 - accuracy: 0.9983 - val_loss: 1.6940e-04 - val_accuracy: 1.0000
Epoch 19/100
88/88 [==============================] - 22s 246ms/step - loss: 0.0052 - accuracy: 0.9991 - val_loss: 1.3579e-04 - val_accuracy: 1.0000
Epoch 20/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0062 - accuracy: 0.9984 - v

In [48]:
################################################################################################################################################
#60% Missing Data SNPs
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_disc=np.array(Traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(X.shape[1]*X.shape[2]*(missD_perc/100))
for i in range(X.shape[0]):
    indices_2d = np.random.choice(X.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(X.shape[2], size=missD, replace=True)
    X[i, indices_2d, indices_3d] = 0

In [49]:
################################################################################################################################################
#100 discrete, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test  = train_test_split(y,X,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_disc_subset(ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_SNPs_60_Trained_Comb_Model_100discrete_50SNPs.mod')

Model: "model_62"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_38 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_39 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_111 (Conv1D)             (None, 98, 256)      23040       input_38[0][0]                   
__________________________________________________________________________________________________
conv1d_114 (Conv1D)             (None, 998, 256)     46080       input_39[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0149 - accuracy: 0.9958 - val_loss: 0.0010 - val_accuracy: 0.9999
Epoch 15/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0122 - accuracy: 0.9968 - val_loss: 7.1191e-04 - val_accuracy: 0.9997
Epoch 16/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0085 - accuracy: 0.9980 - val_loss: 6.9365e-04 - val_accuracy: 0.9997
Epoch 17/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0094 - accuracy: 0.9976 - val_loss: 5.6586e-04 - val_accuracy: 0.9997
Epoch 18/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0081 - accuracy: 0.9980 - val_loss: 5.2208e-04 - val_accuracy: 0.9999
Epoch 19/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0075 - accuracy: 0.9978 - val_loss: 5.5363e-04 - val_accuracy: 0.9999
Epoch 20/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0067 - accuracy: 0.9981 - val_l

Epoch 69/100
88/88 [==============================] - 22s 248ms/step - loss: 7.9027e-04 - accuracy: 0.9998 - val_loss: 1.9091e-04 - val_accuracy: 0.9999
Epoch 70/100
88/88 [==============================] - 22s 249ms/step - loss: 8.2336e-04 - accuracy: 0.9998 - val_loss: 1.5758e-04 - val_accuracy: 0.9999
Epoch 71/100
88/88 [==============================] - 22s 249ms/step - loss: 8.4398e-04 - accuracy: 0.9997 - val_loss: 1.0504e-04 - val_accuracy: 0.9999
Epoch 72/100
88/88 [==============================] - 22s 249ms/step - loss: 7.6014e-04 - accuracy: 0.9998 - val_loss: 1.2524e-04 - val_accuracy: 0.9999
Epoch 73/100
88/88 [==============================] - 22s 248ms/step - loss: 8.1756e-04 - accuracy: 0.9998 - val_loss: 2.2669e-04 - val_accuracy: 0.9999
Epoch 74/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0010 - accuracy: 0.9997 - val_loss: 1.3144e-04 - val_accuracy: 0.9999
Epoch 75/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0013 - 

In [50]:
################################################################################################################################################
# 20% Missing Data discrete
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_disc=np.array(Traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(traits_disc.shape[1]*traits_disc.shape[2]*(missD_perc/100))
for i in range(traits_disc.shape[0]):
    indices_2d = np.random.choice(traits_disc.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_disc.shape[2], size=missD, replace=True)
    traits_disc[i, indices_2d, indices_3d] = 0

In [51]:
################################################################################################################################################
#100 discrete, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test  = train_test_split(y,X,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_disc_subset(ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_discrete_20_Trained_Comb_Model_100discrete_50SNPs.mod')

Model: "model_65"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_40 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_41 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_117 (Conv1D)             (None, 98, 256)      23040       input_40[0][0]                   
__________________________________________________________________________________________________
conv1d_120 (Conv1D)             (None, 998, 256)     46080       input_41[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0092 - accuracy: 0.9975 - val_loss: 6.4157e-05 - val_accuracy: 1.0000
Epoch 15/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0081 - accuracy: 0.9981 - val_loss: 4.4645e-05 - val_accuracy: 1.0000
Epoch 16/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0063 - accuracy: 0.9985 - val_loss: 2.8859e-05 - val_accuracy: 1.0000
Epoch 17/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0064 - accuracy: 0.9983 - val_loss: 2.6790e-05 - val_accuracy: 1.0000
Epoch 18/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0058 - accuracy: 0.9987 - val_loss: 2.0840e-05 - val_accuracy: 1.0000
Epoch 19/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0054 - accuracy: 0.9985 - val_loss: 1.8270e-05 - val_accuracy: 1.0000
Epoch 20/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0058 - accuracy: 0.9985 - v

In [52]:
################################################################################################################################################
#100 discrete
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_disc_train, traits_disc_test  = train_test_split(y,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = disc_subset(ytrain, ytest,traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_discrete_20_Trained_Traits_Model_100discrete.mod')

Model: "model_67"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_42 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_123 (Conv1D)          (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_123 (Bat (None, 98, 256)           1024      
_________________________________________________________________
conv1d_124 (Conv1D)          (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_124 (Bat (None, 96, 256)           1024      
_________________________________________________________________
conv1d_125 (Conv1D)          (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_125 (Bat (None, 94, 256)           102

Epoch 43/100
88/88 [==============================] - 1s 15ms/step - loss: 0.0231 - accuracy: 0.9934 - val_loss: 0.1209 - val_accuracy: 0.9681
Epoch 44/100
88/88 [==============================] - 1s 15ms/step - loss: 0.0221 - accuracy: 0.9936 - val_loss: 0.1400 - val_accuracy: 0.9677
Epoch 45/100
88/88 [==============================] - 1s 15ms/step - loss: 0.0194 - accuracy: 0.9941 - val_loss: 0.1367 - val_accuracy: 0.9677
Epoch 46/100
88/88 [==============================] - 1s 15ms/step - loss: 0.0180 - accuracy: 0.9945 - val_loss: 0.1245 - val_accuracy: 0.9693
Epoch 47/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0182 - accuracy: 0.9946 - val_loss: 0.1489 - val_accuracy: 0.9669
Epoch 48/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0183 - accuracy: 0.9944 - val_loss: 0.1313 - val_accuracy: 0.9693
Epoch 49/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0161 - accuracy: 0.9953 - val_loss: 0.1227 - val_accuracy: 0.9712

In [53]:
################################################################################################################################################
# 40% Missing Data discrete
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_disc=np.array(Traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(traits_disc.shape[1]*traits_disc.shape[2]*(missD_perc/100))
for i in range(traits_disc.shape[0]):
    indices_2d = np.random.choice(traits_disc.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_disc.shape[2], size=missD, replace=True)
    traits_disc[i, indices_2d, indices_3d] = 0

In [54]:
################################################################################################################################################
#100 discrete, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test  = train_test_split(y,X,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_disc_subset(ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_discrete_40_Trained_Comb_Model_100discrete_50SNPs.mod')

Model: "model_70"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_43 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_44 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_126 (Conv1D)             (None, 98, 256)      23040       input_43[0][0]                   
__________________________________________________________________________________________________
conv1d_129 (Conv1D)             (None, 998, 256)     46080       input_44[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 246ms/step - loss: 0.0061 - accuracy: 0.9984 - val_loss: 1.1720e-04 - val_accuracy: 0.9999
Epoch 15/100
88/88 [==============================] - 22s 246ms/step - loss: 0.0055 - accuracy: 0.9985 - val_loss: 6.4456e-05 - val_accuracy: 1.0000
Epoch 16/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0045 - accuracy: 0.9990 - val_loss: 6.0559e-05 - val_accuracy: 1.0000
Epoch 17/100
88/88 [==============================] - 22s 247ms/step - loss: 0.0046 - accuracy: 0.9990 - val_loss: 8.3800e-05 - val_accuracy: 1.0000
Epoch 18/100
88/88 [==============================] - 22s 246ms/step - loss: 0.0039 - accuracy: 0.9988 - val_loss: 1.2529e-04 - val_accuracy: 0.9999
Epoch 19/100
88/88 [==============================] - 22s 246ms/step - loss: 0.0030 - accuracy: 0.9995 - val_loss: 1.1926e-04 - val_accuracy: 0.9999
Epoch 20/100
88/88 [==============================] - 22s 246ms/step - loss: 0.0041 - accuracy: 0.9991 - v

In [55]:
################################################################################################################################################
#100 discrete
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_disc_train, traits_disc_test  = train_test_split(y,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = disc_subset(ytrain, ytest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_discrete_40_Trained_Traits_Model_100discrete.mod')

Model: "model_72"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_45 (InputLayer)        [(None, 100, 30)]         0         
_________________________________________________________________
conv1d_132 (Conv1D)          (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization_132 (Bat (None, 98, 256)           1024      
_________________________________________________________________
conv1d_133 (Conv1D)          (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_133 (Bat (None, 96, 256)           1024      
_________________________________________________________________
conv1d_134 (Conv1D)          (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_134 (Bat (None, 94, 256)           102

Epoch 43/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0798 - accuracy: 0.9729 - val_loss: 0.4515 - val_accuracy: 0.8772
Epoch 44/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0696 - accuracy: 0.9770 - val_loss: 0.4776 - val_accuracy: 0.8741
Epoch 45/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0670 - accuracy: 0.9762 - val_loss: 0.4709 - val_accuracy: 0.8769
Epoch 46/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0641 - accuracy: 0.9786 - val_loss: 0.5326 - val_accuracy: 0.8629
Epoch 47/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0564 - accuracy: 0.9812 - val_loss: 0.4842 - val_accuracy: 0.8765
Epoch 48/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0540 - accuracy: 0.9823 - val_loss: 0.5057 - val_accuracy: 0.8747
Epoch 49/100
88/88 [==============================] - 1s 16ms/step - loss: 0.0521 - accuracy: 0.9830 - val_loss: 0.5388 - val_accuracy: 0.8691

In [8]:
################################################################################################################################################
# 60% Missing Data discrete
################################################################################################################################################
# Load the original SNPs matrices
X=np.array(u)

# Load the original traits matrices
traits_disc=np.array(Traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(traits_disc.shape[1]*traits_disc.shape[2]*(missD_perc/100))
for i in range(traits_disc.shape[0]):
    indices_2d = np.random.choice(traits_disc.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_disc.shape[2], size=missD, replace=True)
    traits_disc[i, indices_2d, indices_3d] = 0

In [57]:
################################################################################################################################################
#100 discrete, 1,000 SNPs
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test  = train_test_split(y,X,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = combined_disc_subset(ytrain, ytest, xtrain, xtest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_discrete_60_Trained_Comb_Model_100discrete_50SNPs.mod')

Model: "model_75"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_46 (InputLayer)           [(None, 100, 30)]    0                                            
__________________________________________________________________________________________________
input_47 (InputLayer)           [(None, 1000, 60)]   0                                            
__________________________________________________________________________________________________
conv1d_135 (Conv1D)             (None, 98, 256)      23040       input_46[0][0]                   
__________________________________________________________________________________________________
conv1d_138 (Conv1D)             (None, 998, 256)     46080       input_47[0][0]                   
___________________________________________________________________________________________

Epoch 14/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0073 - accuracy: 0.9984 - val_loss: 3.8312e-05 - val_accuracy: 1.0000
Epoch 15/100
88/88 [==============================] - 22s 248ms/step - loss: 0.0056 - accuracy: 0.9988 - val_loss: 2.3185e-05 - val_accuracy: 1.0000
Epoch 16/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0051 - accuracy: 0.9989 - val_loss: 2.1827e-05 - val_accuracy: 1.0000
Epoch 17/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0044 - accuracy: 0.9990 - val_loss: 2.1624e-05 - val_accuracy: 1.0000
Epoch 18/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0049 - accuracy: 0.9986 - val_loss: 1.2805e-05 - val_accuracy: 1.0000
Epoch 19/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0039 - accuracy: 0.9989 - val_loss: 1.0937e-05 - val_accuracy: 1.0000
Epoch 20/100
88/88 [==============================] - 22s 249ms/step - loss: 0.0046 - accuracy: 0.9989 - v

In [9]:
################################################################################################################################################
#100 discrete
################################################################################################################################################

# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_disc_train, traits_disc_test  = train_test_split(y,traits_disc,test_size=0.25, shuffle=True,stratify=y)

# train the network
model = disc_subset(ytrain, ytest, traits_disc_train, traits_disc_test)

# save the model
model.save(filepath='./Trained_Models/MD_discrete_60_Trained_Traits_Model_100discrete.mod')

Model: "model_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, 100, 30)]         0         
_________________________________________________________________
conv1d (Conv1D)              (None, 98, 256)           23040     
_________________________________________________________________
batch_normalization (BatchNo (None, 98, 256)           1024      
_________________________________________________________________
conv1d_1 (Conv1D)            (None, 96, 256)           196608    
_________________________________________________________________
batch_normalization_1 (Batch (None, 96, 256)           1024      
_________________________________________________________________
conv1d_2 (Conv1D)            (None, 94, 256)           196608    
_________________________________________________________________
batch_normalization_2 (Batch (None, 94, 256)           1024

Epoch 43/100
88/88 [==============================] - 1s 17ms/step - loss: 0.2815 - accuracy: 0.8839 - val_loss: 0.8914 - val_accuracy: 0.6817
Epoch 44/100
88/88 [==============================] - 1s 16ms/step - loss: 0.2676 - accuracy: 0.8912 - val_loss: 0.9638 - val_accuracy: 0.6843
Epoch 45/100
88/88 [==============================] - 1s 17ms/step - loss: 0.2471 - accuracy: 0.9019 - val_loss: 0.9530 - val_accuracy: 0.6911
Epoch 46/100
88/88 [==============================] - 1s 17ms/step - loss: 0.2311 - accuracy: 0.9056 - val_loss: 0.9212 - val_accuracy: 0.7013
Epoch 47/100
88/88 [==============================] - 1s 16ms/step - loss: 0.2198 - accuracy: 0.9126 - val_loss: 0.9381 - val_accuracy: 0.7061
Epoch 48/100
88/88 [==============================] - 1s 16ms/step - loss: 0.2017 - accuracy: 0.9191 - val_loss: 0.9701 - val_accuracy: 0.6969
Epoch 49/100
88/88 [==============================] - 1s 16ms/step - loss: 0.1901 - accuracy: 0.9250 - val_loss: 1.0060 - val_accuracy: 0.7032

88/88 [==============================] - 1s 16ms/step - loss: 0.0325 - accuracy: 0.9883 - val_loss: 1.8380 - val_accuracy: 0.7275
Time: 148.60287165641785
INFO:tensorflow:Assets written to: ./Trained_Models/MD_discrete_60_Trained_Traits_Model_100discrete.mod/assets


In [29]:
# Now that the models are trained, we will evaluate their accuracy based on the test set. For that, we will build confusion matrices containing the true and predicted scenarions for each simulation on the test set.

# first import the libraries
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
from keras.models import load_model
from sklearn.metrics import confusion_matrix

# load the trained models.
model1 = load_model('./Trained_Models/MD_BM_20_Trained_Traits_Model_100BM.mod')
model2 = load_model('./Trained_Models/MD_OU_20_Trained_Traits_Model_100OU.mod')
model3 = load_model('./Trained_Models/MD_discrete_20_Trained_Traits_Model_100discrete.mod')
model4 = load_model('./Trained_Models/MD_SNPs_20_Trained_CNN_Model_50SNPs.mod')
model5 = load_model('./Trained_Models/MD_SNPs_20_Trained_Comb_Model_100BM_50SNPs.mod')
model6 = load_model('./Trained_Models/MD_SNPs_20_Trained_Comb_Model_100OU_50SNPs.mod')
model7 = load_model('./Trained_Models/MD_SNPs_20_Trained_Comb_Model_100discrete_50SNPs.mod')
model8 = load_model('./Trained_Models/MD_BM_20_Trained_Comb_Model_100BM_50SNPs.mod')
model9 = load_model('./Trained_Models/MD_OU_20_Trained_Comb_Model_100OU_50SNPs.mod')
model10 = load_model('./Trained_Models/MD_discrete_20_Trained_Comb_Model_100discrete_50SNPs.mod')
model11 = load_model('./Trained_Models/MD_BM_40_Trained_Traits_Model_100BM.mod')
model12 = load_model('./Trained_Models/MD_OU_40_Trained_Traits_Model_100OU.mod')
model13 = load_model('./Trained_Models/MD_discrete_40_Trained_Traits_Model_100discrete.mod')
model14 = load_model('./Trained_Models/MD_SNPs_40_Trained_CNN_Model_50SNPs.mod')
model15 = load_model('./Trained_Models/MD_SNPs_40_Trained_Comb_Model_100BM_50SNPs.mod')
model16 = load_model('./Trained_Models/MD_SNPs_40_Trained_Comb_Model_100OU_50SNPs.mod')
model17 = load_model('./Trained_Models/MD_SNPs_40_Trained_Comb_Model_100discrete_50SNPs.mod')
model18 = load_model('./Trained_Models/MD_BM_40_Trained_Comb_Model_100BM_50SNPs.mod')
model19 = load_model('./Trained_Models/MD_OU_40_Trained_Comb_Model_100OU_50SNPs.mod')
model20 = load_model('./Trained_Models/MD_discrete_40_Trained_Comb_Model_100discrete_50SNPs.mod')
model21 = load_model('./Trained_Models/MD_BM_60_Trained_Traits_Model_100BM.mod')
model22 = load_model('./Trained_Models/MD_OU_60_Trained_Traits_Model_100OU.mod')
model23 = load_model('./Trained_Models/MD_discrete_60_Trained_Traits_Model_100discrete.mod')
model24 = load_model('./Trained_Models/MD_SNPs_60_Trained_CNN_Model_50SNPs.mod')
model25 = load_model('./Trained_Models/MD_SNPs_60_Trained_Comb_Model_100BM_50SNPs.mod')
model26 = load_model('./Trained_Models/MD_SNPs_60_Trained_Comb_Model_100OU_50SNPs.mod')
model27 = load_model('./Trained_Models/MD_SNPs_60_Trained_Comb_Model_100discrete_50SNPs.mod')
model28 = load_model('./Trained_Models/MD_BM_60_Trained_Comb_Model_100BM_50SNPs.mod')
model29 = load_model('./Trained_Models/MD_OU_60_Trained_Comb_Model_100OU_50SNPs.mod')
model30 = load_model('./Trained_Models/MD_discrete_60_Trained_Comb_Model_100discrete_50SNPs.mod')

In [29]:
#remove
# load the traits simulated under the BM model for the 3 scenarios. 
traits_BM = []
traits_BM = np.loadtxt("./traits/traits_BM.txt").reshape(30000,-1,100)
# transform into a NumPy array. 
traits_BM = np.array(traits_BM)

# standard scale the continuous (BM) traits
scalers_BM = {}
for i in range(traits_BM.shape[2]):
    scalers_BM[i] = StandardScaler(copy=False)
    traits_BM[:, :, i] = scalers_BM[i].fit_transform(traits_BM[:, :, i]) 

In [ ]:
#remove
# load the traits simulated under the OU model for the 3 scenarios. 
traits_OU = []
traits_OU = np.loadtxt("./traits/traits_OU.txt").reshape(30000,-1,100)
# transform into a NumPy array. 
traits_OU = np.array(traits_OU)

# standard scale the continuous (OU) traits
scalers_OU = {}
for i in range(traits_OU.shape[2]):
    scalers_OU[i] = StandardScaler(copy=False)
    traits_OU[:, :, i] = scalers_OU[i].fit_transform(traits_OU[:, :, i])

In [30]:
# load the traits simulated under the BM model for the 3 scenarios.
traits_BM = []
traits_BM = np.loadtxt("./testSims/traits/traits_BM.txt").reshape(3000,-1,100)
# transform into a NumPy array. 
traits_BM = np.array(traits_BM)

#Standard scale the continuous traits.
for i in range(traits_BM.shape[2]):
    traits_BM[:, :, i] = scalers_BM[i].transform(traits_BM[:, :, i]) 

# transform into a NumPy array. 
Traits_BM = np.array(traits_BM)
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(traits_BM.shape[1]*traits_BM.shape[2]*(missD_perc/100))
for i in range(traits_BM.shape[0]):
    indices_2d = np.random.choice(traits_BM.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_BM.shape[2], size=missD, replace=True)
    traits_BM[i, indices_2d, indices_3d] = 0

# transform into a NumPy array.
traits_BM20=np.array(traits_BM)
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(traits_BM.shape[1]*traits_BM.shape[2]*(missD_perc/100))
for i in range(traits_BM.shape[0]):
    indices_2d = np.random.choice(traits_BM.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_BM.shape[2], size=missD, replace=True)
    traits_BM[i, indices_2d, indices_3d] = 0

# transform into a NumPy array.
traits_BM40=np.array(traits_BM)
traits_BM=np.array(Traits_BM)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(traits_BM.shape[1]*traits_BM.shape[2]*(missD_perc/100))
for i in range(traits_BM.shape[0]):
    indices_2d = np.random.choice(traits_BM.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_BM.shape[2], size=missD, replace=True)
    traits_BM[i, indices_2d, indices_3d] = 0

# transform into a NumPy array.
traits_BM60=np.array(traits_BM)
traits_BM=np.array(Traits_BM)

# load the traits simulated under the OU model for the 3 scenarios.
traits_OU = []
traits_OU = np.loadtxt("./testSims/traits/traits_OU.txt").reshape(3000,-1,100)
# transform into a NumPy array. 
traits_OU = np.array(traits_OU)

#Standard scale the continuous variables
for i in range(traits_OU.shape[2]):
    traits_OU[:, :, i] = scalers_OU[i].transform(traits_OU[:, :, i]) 

# transform into a NumPy array. 
Traits_OU = np.array(traits_OU)
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(traits_OU.shape[1]*traits_OU.shape[2]*(missD_perc/100))
for i in range(traits_OU.shape[0]):
    indices_2d = np.random.choice(traits_OU.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_OU.shape[2], size=missD, replace=True)
    traits_OU[i, indices_2d, indices_3d] = 0

# transform into a NumPy array. 
traits_OU20=np.array(traits_OU)
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(traits_OU.shape[1]*traits_OU.shape[2]*(missD_perc/100))
for i in range(traits_OU.shape[0]):
    indices_2d = np.random.choice(traits_OU.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_OU.shape[2], size=missD, replace=True)
    traits_OU[i, indices_2d, indices_3d] = 0

# transform into a NumPy array. 
traits_OU40=np.array(traits_OU)
traits_OU=np.array(Traits_OU)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(traits_OU.shape[1]*traits_OU.shape[2]*(missD_perc/100))
for i in range(traits_OU.shape[0]):
    indices_2d = np.random.choice(traits_OU.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_OU.shape[2], size=missD, replace=True)
    traits_OU[i, indices_2d, indices_3d] = 0

# transform into a NumPy array. 
traits_OU60=np.array(traits_OU)
traits_OU=np.array(Traits_OU)

# load the discrete traits simulated for the 3 scenarios.
traits_disc = []
traits_disc = np.loadtxt("./testSims/traits/traits_disc.txt").reshape(3000,-1,100)

# transform into a NumPy array. 
traits_disc = np.array(traits_disc)
traits_disc = np.array(traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(traits_disc.shape[1]*traits_disc.shape[2]*(missD_perc/100))
for i in range(traits_disc.shape[0]):
  for m in range(missD):
    j = random.randint(0, traits_disc.shape[1] - 1)
    k = random.randint(0, traits_disc.shape[2] - 1)
    traits_disc[i][j][k] = 0

# transform into a NumPy array. 
traits_disc20=np.array(traits_disc)
traits_disc=np.array(traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(traits_disc.shape[1]*traits_disc.shape[2]*(missD_perc/100))
for i in range(traits_disc.shape[0]):
    indices_2d = np.random.choice(traits_disc.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_disc.shape[2], size=missD, replace=True)
    traits_disc[i, indices_2d, indices_3d] = 0

# transform into a NumPy array. 
traits_disc40=np.array(traits_disc)
traits_disc=np.array(traits_disc)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(traits_disc.shape[1]*traits_disc.shape[2]*(missD_perc/100))
for i in range(traits_disc.shape[0]):
    indices_2d = np.random.choice(traits_disc.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(traits_disc.shape[2], size=missD, replace=True)
    traits_disc[i, indices_2d, indices_3d] = 0

# transform into a NumPy array. 
traits_disc60=np.array(traits_disc)
traits_disc=np.array(traits_disc)

# load the SNPs simulated for the 3 scenarios. 
u1 = np.load("./testSims/Model_1sp.npz",mmap_mode='r')
u2 = np.load("./testSims/Model_2sp.npz",mmap_mode='r')
u3 = np.load("./testSims/Model_3sp.npz",mmap_mode='r')

# combine the loaded SNPs in a single NumPy array.
Xtest=np.concatenate((u1['Model_1sp'],u2['Model_2sp'],u3['Model_3sp']),axis=0)

#transform major alleles in -1 and minor in 1
for arr,array in enumerate(Xtest):
  for idx,row in enumerate(array):
    if np.count_nonzero(row) > len(row)/2:
      Xtest[arr][idx][Xtest[arr][idx] == 1] = -1
      Xtest[arr][idx][Xtest[arr][idx] == 0] = 1
    else:
      Xtest[arr][idx][Xtest[arr][idx] == 0] = -1

# transform into a NumPy array. 
xtest=np.array(Xtest)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 20
missD = int(xtest.shape[1]*xtest.shape[2]*(missD_perc/100))
for i in range(xtest.shape[0]):
    indices_2d = np.random.choice(xtest.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(xtest.shape[2], size=missD, replace=True)
    xtest[i, indices_2d, indices_3d] = 0

# transform into a NumPy array. 
xtest20=np.array(xtest)
xtest=np.array(Xtest)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 40
missD = int(xtest.shape[1]*xtest.shape[2]*(missD_perc/100))
for i in range(xtest.shape[0]):
    indices_2d = np.random.choice(xtest.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(xtest.shape[2], size=missD, replace=True)
    xtest[i, indices_2d, indices_3d] = 0


# transform into a NumPy array. 
xtest40=np.array(xtest)
xtest=np.array(Xtest)

#Add missing data as 0s, according to a specific missing data percentage
missD_perc = 60
missD = int(xtest.shape[1]*xtest.shape[2]*(missD_perc/100))
for i in range(xtest.shape[0]):
    indices_2d = np.random.choice(xtest.shape[1], size=missD, replace=True)
    indices_3d = np.random.choice(xtest.shape[2], size=missD, replace=True)
    xtest[i, indices_2d, indices_3d] = 0


# transform into a NumPy array. 
xtest60=np.array(xtest)
xtest=np.array(Xtest)

# create a label vector in the same order as the simulations.  
ytest=[0 for i in range(len(u1['Model_1sp']))]
ytest.extend([1 for i in range(len(u2['Model_2sp']))])
ytest.extend([2 for i in range(len(u3['Model_3sp']))])
ytest = np.array(ytest)

In [31]:
#define a funtion to build the confusion matrix
def makeConfusionMatrixHeatmap(data, title, trueClassOrderLs, predictedClassOrderLs, ax):
    data = np.array(data)
    data = normalize(data, axis=1, norm='l1')
    heatmap = ax.pcolor(data, cmap=plt.cm.Blues, vmin=0.0, vmax=1.0)

    for i in range(len(predictedClassOrderLs)):
        for j in reversed(range(len(trueClassOrderLs))):
            val = 100*data[j, i]
            if val > 50:
                c = '0.9'
            else:
                c = 'black'
            ax.text(i + 0.5, j + 0.5, '%.2f%%' % val, horizontalalignment='center', verticalalignment='center', color=c, fontsize=9)

    cbar = plt.colorbar(heatmap, cmap=plt.cm.Blues, ax=ax)
    cbar.set_label("Fraction of simulations assigned to class", rotation=270, labelpad=20, fontsize=11)

    # put the major ticks at the middle of each cell
    ax.set_xticks(np.arange(data.shape[1]) + 0.5, minor=False)
    ax.set_yticks(np.arange(data.shape[0]) + 0.5, minor=False)
    ax.axis('tight')
    ax.set_title(title)

    #labels
    ax.set_xticklabels(predictedClassOrderLs, minor=False, fontsize=9, rotation=45)
    ax.set_yticklabels(reversed(trueClassOrderLs), minor=False, fontsize=9)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")

In [ ]:
# Now we will plot the confusion matrices for each trained model
#first get the predictions
pred_all = np.array([])
pred = model1.predict(np.swapaxes(traits_BM20, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/BM20_pred_all.csv', pred, delimiter=',')
counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()
classOrderLs=['one','two','three']

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions for the next dataset
pred = model2.predict(np.swapaxes(traits_OU20, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/OU20_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model3.predict(np.swapaxes(traits_disc20, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/disc20_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model4.predict(xtest20)
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/SNP20_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model5.predict([np.swapaxes(traits_BM, 1, 2),  xtest20])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/BM_SNP20_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model6.predict([np.swapaxes(traits_OU, 1, 2),  xtest20])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/OU_SNP20_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model7.predict([np.swapaxes(traits_disc, 1, 2),  xtest20])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/disc_SNP20_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model8.predict([np.swapaxes(traits_BM20, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/BM20_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model9.predict([np.swapaxes(traits_OU20, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/OU20_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model10.predict([np.swapaxes(traits_disc20, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/disc20_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model11.predict(np.swapaxes(traits_BM40, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/BM40_pred_all.csv', pred, delimiter=',')
print (confusion_matrix(ytest, pred_cat))

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()
classOrderLs=['one','two','three']

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model12.predict(np.swapaxes(traits_OU40, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/OU40_pred_all.csv', pred, delimiter=',')


counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model13.predict(np.swapaxes(traits_disc40, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/disc40_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model14.predict(xtest40)
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/SNP40_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model15.predict([np.swapaxes(traits_BM, 1, 2),  xtest40])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/BM_SNP40_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

#now the actual work #6
#first get the predictions
pred = model16.predict([np.swapaxes(traits_OU, 1, 2),  xtest40])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/OU_SNP40_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model17.predict([np.swapaxes(traits_disc, 1, 2),  xtest40])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/disc_SNP40_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model18.predict([np.swapaxes(traits_BM40, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/BM40_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model19.predict([np.swapaxes(traits_OU40, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/OU40_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model20.predict([np.swapaxes(traits_disc40, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/disc40_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model21.predict(np.swapaxes(traits_BM60, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/BM60_pred_all.csv', pred, delimiter=',')
print (confusion_matrix(ytest, pred_cat))

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()
classOrderLs=['one','two','three']

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model22.predict(np.swapaxes(traits_OU60, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/OU60_pred_all.csv', pred, delimiter=',')


counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model23.predict(np.swapaxes(traits_disc60, 1, 2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/disc60_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model24.predict(xtest60)
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/SNP60_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model25.predict([np.swapaxes(traits_BM, 1, 2),  xtest60])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/BM_SNP60_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model26.predict([np.swapaxes(traits_OU, 1, 2),  xtest60])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/OU_SNP60_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model27.predict([np.swapaxes(traits_disc, 1, 2),  xtest60])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/disc_SNP60_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model28.predict([np.swapaxes(traits_BM60, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/BM60_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model29.predict([np.swapaxes(traits_OU60, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/OU60_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()

# get the predictions
pred = model30.predict([np.swapaxes(traits_disc60, 1, 2),  xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
pred_all = pred[np.arange(pred.shape[0]), np.array([i.max(0) for i in ytest])]

np.savetxt('./testSims/Predictions/MissingData/pred_MissingValues/disc60_SNP_pred_all.csv', pred, delimiter=',')

counts=[[0.,0.,0.],[0.,0.,0.],[0.,0.,0.]]
for i in range(len(ytest_cat)):
    counts[ytest[i]][pred_cat[i]] += 1
counts.reverse()

#now do the plotting
fig,ax= plt.subplots(1,1)
makeConfusionMatrixHeatmap(counts, "Confusion matrix", classOrderLs, classOrderLs, ax)
plt.show()